In [1]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
import os, json

IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32 
DATASET_PATH = "../PlantVillage" 

train_datagen = ImageDataGenerator(rescale=1./255, validation_split=0.2)
val_datagen = ImageDataGenerator(rescale=1./255, validation_split=0.2)

train_generator = train_datagen.flow_from_directory(
    DATASET_PATH, target_size=IMAGE_SIZE, batch_size=BATCH_SIZE, class_mode='categorical', subset='training'
)

val_generator = val_datagen.flow_from_directory(
    DATASET_PATH, target_size=IMAGE_SIZE, batch_size=BATCH_SIZE, class_mode='categorical', subset='validation'
)

os.makedirs("../models", exist_ok=True)
class_names = sorted(list(train_generator.class_indices.keys()))
with open('../models/class_indices.json', 'w') as f:
    json.dump(class_names, f)

Found 16516 images belonging to 15 classes.
Found 4122 images belonging to 15 classes.


In [2]:
base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base_model.trainable = False

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(256, activation='relu')(x)
x = Dropout(0.5)(x)
output_layer = Dense(train_generator.num_classes, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=output_layer)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), 
    loss='categorical_crossentropy', 
    metrics=['accuracy']
)

In [ ]:
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=5
)

model.save('..models/plant_model_v1.h5')

Epoch 1/3
131/517 ━━━━━━━━━━━━━━━━━━━━ 2:21 366ms/step - accuracy: 0.4553 - loss: 1.7915

KeyboardInterrupt: 